In [ ]:
# Run this first to clone the published DSA implementation.
from pathlib import Path
import subprocess
import sys
import os
import json
import hashlib
import shutil

REPO_URL = "https://github.com/sttruji/dynamic-state-anchoring.git"
REPO_REF = "main"  # Set a commit SHA here to reproduce a particular implementation.
REPO_DIR = Path("/content/dynamic-state-anchoring")

def git_output(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

if not REPO_DIR.exists():
    try:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Check that the repository is reachable, then rerun. "
                           "Private repositories need authenticated Git access configured separately.") from exc
else:
    remote = git_output("remote", "get-url", "origin").rstrip("/").removesuffix(".git")
    if remote != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError("Existing checkout has a different remote; choose another REPO_DIR.")
    if git_output("status", "--porcelain"):
        raise RuntimeError("Existing checkout has local changes. Preserve them before updating it.")
subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
REPO_COMMIT = git_output("rev-parse", "HEAD")
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError("This ref does not contain the DSA implementation.")
print("DSA implementation:", REPO_COMMIT)


# Dynamic State Anchoring · K=8 and FLT-inspired memory injection

Select **T4 GPU → Run all** in this current GitHub notebook. A fresh experiment
runs four matched conditions: **standard/static, standard/learned,
FLT-inspired/static, FLT-inspired/learned**. Each trains on the same 32 questions
for 200 updates with **8 latent loops** and the same initial parameters.
The K=16 pilot is preserved separately; new code does not reuse its checkpoints.
Cached corpus/model downloads are reused when available.

The FLT-inspired GPT-2 variant resets the current residual stream each loop and
uses the previous five memory vectors as attention queries at every block.
Question positions keep ordinary causal self-attention; current-layer states
supply keys/values. Existing projections are reused, with no added parameters
and no detached recurrent gradients. The explicit teacher and answer decoder
keep ordinary GPT-2 computation. This adapts [FLT](https://arxiv.org/html/2605.18797v1)
to our fixed memory; it is not a full reproduction or a proven stability guarantee.

A bounded gradient screen checks one fixed training row at K=1/8/16 before
training. Finite gradients are required; gradient improvement is recorded, not
assumed. Training and raw residual/gradient diagnostics then assess all four
conditions. K=16 probes extrapolate beyond the K=8 training budget.

Each completed arm saves one full archive to Drive and requests its own browser
download before the next arm. **Keep the four checkpoint ZIPs and wait for their
downloads to finish before disconnecting.** The last cell downloads one small
review ZIP to hand back. Mounted Drive readback cannot prove cloud durability.

`auto_pilot` reuses fully verified completed arms and restarts partial/missing
arms. `restart_all` forces a fresh run of all four. Validation is for development;
the held-out test remains disabled. No larger run starts automatically.

In [ ]:
# Reuse Colab's CUDA/PyTorch stack; isolate project packages from the notebook kernel.
# The helper uses --without-pip --system-site-packages and targets this environment explicitly.
VENV_DIR = Path("/content/dsa-venv")
subprocess.run([sys.executable, str(REPO_DIR / "scripts/setup_colab.py"),
                "--venv-dir", str(VENV_DIR),
                "--requirements", str(REPO_DIR / "requirements-colab.txt")], check=True)
PYTHON = str(VENV_DIR / "bin/python")
print("Project Python:", PYTHON)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = Path("/content/drive/MyDrive/dynamic-state-anchoring")
for name in ("corpus", "prepared", "runs", "reports", "cache/huggingface"):
    (PROJECT_DIR / name).mkdir(parents=True, exist_ok=True)
# Set before any subprocess imports Hugging Face packages.
os.environ["HF_HOME"] = str(PROJECT_DIR / "cache/huggingface")
os.environ["HF_HUB_CACHE"] = str(PROJECT_DIR / "cache/huggingface/hub")
os.environ["HUGGINGFACE_HUB_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TRANSFORMERS_CACHE"] = os.environ["HF_HUB_CACHE"]
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
print("Persistent storage:", PROJECT_DIR)


In [ ]:
# Default: reuse finished compatible arms; restart missing/partial arms from scratch.
RUN_MODE = "auto_pilot"  # "restart_all" retrains both; "train_pilot" selects NEW_RUN_NAME.
EXISTING_RUN_NAME = f"gpt2-k8-flt-v1-{REPO_COMMIT[:12]}"
NEW_RUN_NAME = f"gpt2-k8-flt-v1-{REPO_COMMIT[:12]}"
EXPERIMENT = "gpt2-k8-flt-v1"
BACKBONE = "yingfanbot/gsm-cot-gpt2"
MODEL_REVISION = "e0ef1887404adbf6c531d2cac7804694321f87bd"
DATA_REVISION = "e06a32ee5e4cd117171daeb4755d2a97ece62761"
SEED, TRAIN_LIMIT, LOOPS, WORKSPACE, STEPS = 7, 32, 8, 4, 200
LEARNING_RATE, SAVE_EVERY, KEEP_CHECKPOINTS, GRADIENT_ACCUMULATION = 3e-5, 25, 2, 1
CONDITIONS = {
    "static": {"recurrence_mode": "standard", "write_mode": "static"},
    "learned": {"recurrence_mode": "standard", "write_mode": "learned"},
    "flt_static": {"recurrence_mode": "flt_memory", "write_mode": "static"},
    "flt_learned": {"recurrence_mode": "flt_memory", "write_mode": "learned"},
}
ARMS = ["static", "learned", "flt_static", "flt_learned"]
SCREEN_LIMIT, SCREEN_DEPTHS = 1, [1, 8, 16]
VALID_LIMIT = 128  # 0 evaluates all 500; test remains disabled.
MAX_NEW_TOKENS = 64
DEVICE = "cuda"
GRADIENT_LIMIT, PROBE_LIMIT, CONTROL_LIMIT = 4, 8, 8
DEPTH_GRID = [1, 2, 4, 8, 16]
COT_MAX_NEW_TOKENS = 192
DOWNLOAD_FULL_CHECKPOINTS = True  # Keep this independent backup on your computer.
assert RUN_MODE in {"auto_pilot", "restart_all", "train_pilot", "review_existing"}
RUN_NAME = NEW_RUN_NAME if RUN_MODE == "train_pilot" else EXISTING_RUN_NAME
assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME
assert ARMS and len(set(ARMS)) == len(ARMS) and set(ARMS) <= set(CONDITIONS)
LEGACY_RUN_ROOT = PROJECT_DIR / "runs" / RUN_NAME
RUN_ROOT = LEGACY_RUN_ROOT if RUN_MODE == "review_existing" else Path("/content/dsa-pilot-working") / RUN_NAME
REPORT_ROOT = PROJECT_DIR / "reports" / RUN_NAME
SMOKE_REPORT = REPORT_ROOT / "unmodified-checkpoint-smoke.json"
CHECKPOINT_BACKUP_DIR = PROJECT_DIR / "bundles" / RUN_NAME / "checkpoints"
TEMPLATE_RUN_DIR = None
saved = {}
for arm in ARMS:
    path = LEGACY_RUN_ROOT / arm / "run_manifest.json"
    if path.is_file():
        try:
            metadata = json.loads(path.read_text())
            contract = metadata["contract"]
            for key in ("backbone", "revision", "loops", "workspace", "seed", "learning_rate", "gradient_accumulation"):
                contract["settings"][key]
            contract["prepared_manifest_sha256"], contract["record_count"]
            saved[arm] = metadata
        except (OSError, ValueError, KeyError, TypeError): print(f"{arm}: unreadable old metadata; automatic restart remains available.")
PREPARED_DRIVE = PROJECT_DIR / "prepared" / f"{REPO_COMMIT[:12]}-train{TRAIN_LIMIT}-k{LOOPS}-s{SEED}"
if saved and RUN_MODE in {"auto_pilot", "restart_all"}:
    original_arm = next(iter(saved))
    original = saved[original_arm]
    TEMPLATE_RUN_DIR = LEGACY_RUN_ROOT / original_arm
    settings = original["contract"]["settings"]
    BACKBONE, MODEL_REVISION = settings["backbone"], settings["revision"]
    LOOPS, WORKSPACE, SEED = settings["loops"], settings["workspace"], settings["seed"]
    LEARNING_RATE, GRADIENT_ACCUMULATION = settings["learning_rate"], settings["gradient_accumulation"]
    TRAIN_LIMIT = original["contract"]["record_count"]
    expected_hash = original["contract"]["prepared_manifest_sha256"]
    candidates = sorted(path.parent for path in (PROJECT_DIR / "prepared").glob("*/manifest.json")
                        if hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash)
    if candidates: PREPARED_DRIVE = candidates[0]
if RUN_MODE == "review_existing":
    saved = {arm: json.loads((RUN_ROOT / arm / "run_manifest.json").read_text()) for arm in ARMS}
    hashes = {m["contract"]["prepared_manifest_sha256"] for m in saved.values()}
    assert len(hashes) == 1, "Selected arms used different prepared datasets."
    expected_hash = next(iter(hashes))
    candidates = sorted(path.parent for path in (PROJECT_DIR / "prepared").glob("*/manifest.json")
                        if hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash)
    if not candidates:
        raise FileNotFoundError("Original prepared Drive cache is missing; restore it before reviewing.")
    PREPARED_DRIVE = candidates[0]
    settings = saved[ARMS[0]]["contract"]["settings"]
    BACKBONE, MODEL_REVISION = settings["backbone"], settings["revision"]
    LOOPS, WORKSPACE, SEED = settings["loops"], settings["workspace"], settings["seed"]
    TRAIN_LIMIT = saved[ARMS[0]]["contract"]["record_count"]
    LEARNING_RATE = settings["learning_rate"]
    review_steps = {}
    for arm in ARMS:
        run_dir = (RUN_ROOT / arm).resolve()
        latest = json.loads((run_dir / "latest.json").read_text())
        step = int(latest["step"])
        checkpoint = (run_dir / latest["path"]).resolve()
        if checkpoint.parent != run_dir or checkpoint.name != f"checkpoint-{step:08d}":
            raise RuntimeError(f"{arm}: latest.json has an invalid checkpoint path/step.")
        completion = checkpoint / "COMPLETE.json"
        if not completion.is_file():
            raise RuntimeError(f"{arm}: latest step {step} has no COMPLETE.json: {checkpoint}")
        marker = json.loads(completion.read_text())
        required = ["training_state.pt", "model/anchor.pt", "model/anchor_config.json", "model/backbone/config.json"]
        if marker.get("schema_version") != 1 or marker.get("step") != step or any(not (checkpoint / name).is_file() for name in required) or not (checkpoint / "model/tokenizer").is_dir():
            raise RuntimeError(f"{arm}: checkpoint step {step} has inconsistent completion metadata or missing files.")
        review_steps[arm] = step
        print(f"{arm}: selected completed checkpoint step {step}; manifest status={saved[arm].get('status')}, recorded steps={saved[arm].get('completed_steps')}")
    STEPS = review_steps[ARMS[0]]
    if len(set(review_steps.values())) != 1:
        print("WARNING: arms have different completed checkpoint steps; the review will flag this comparison.")
PREP_NAME = PREPARED_DRIVE.name
PREPARED_LOCAL = Path("/content/dsa-prepared") / PREP_NAME
EVAL_BUDGETS = [LOOPS]
review_settings = {key: globals()[key] for key in ("RUN_MODE", "BACKBONE", "MODEL_REVISION", "SEED", "TRAIN_LIMIT",
    "LOOPS", "WORKSPACE", "STEPS", "LEARNING_RATE", "GRADIENT_ACCUMULATION", "ARMS", "VALID_LIMIT", "MAX_NEW_TOKENS",
    "DEVICE", "GRADIENT_LIMIT", "PROBE_LIMIT", "CONTROL_LIMIT", "DEPTH_GRID", "COT_MAX_NEW_TOKENS", "CONDITIONS", "SCREEN_LIMIT", "SCREEN_DEPTHS")}
settings_hash = hashlib.sha256(json.dumps(review_settings, sort_keys=True).encode()).hexdigest()[:8]
REVIEW_NAME = f"pilot-review-v4-{REPO_COMMIT[:12]}-{settings_hash}"
BASE_REVIEW_NAME = REVIEW_NAME
REVIEW_ROOT = REPORT_ROOT / REVIEW_NAME
REVIEW_ROOT.mkdir(parents=True, exist_ok=True)
import runpy, datetime
_run_logged = runpy.run_path(str(REPO_DIR / "scripts/colab_process.py"))["run_logged"]
def run_project(*args):
    return _run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR,
                       env=os.environ.copy(), log_dir=REVIEW_ROOT / "logs")
status_path = REVIEW_ROOT / "session_status.json"
STAGE_STATUS = json.loads(status_path.read_text()) if status_path.exists() else {}
def record_stage(name, status, **details):
    STAGE_STATUS[name] = {"status": status, "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), **details}
    temporary = status_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(STAGE_STATUS, indent=2) + "\n")
    temporary.replace(status_path)
def stage(name, *args):
    try:
        run_project(*args)
    except Exception as error:
        record_stage(name, "failed", error=str(error))
        print(f"STAGE FAILED: {name}. Error saved; continuing toward the review ZIP.", flush=True)
        return False
    record_stage(name, "passed")
    return True
config = {"RUN_NAME": RUN_NAME, "REVIEW_NAME": REVIEW_NAME, "REPO_COMMIT": REPO_COMMIT,
          "DATA_REVISION": DATA_REVISION, "EXPERIMENT": EXPERIMENT, **review_settings}
protocol_path = REVIEW_ROOT / "protocol.json"
protocol_path.write_text(json.dumps(config, indent=2) + "\n")
print(json.dumps(config, indent=2))
print("Working checkpoints:", RUN_ROOT)
print("Full Drive backups:", CHECKPOINT_BACKUP_DIR)
print("Review outputs:", REVIEW_ROOT)
print("Small-pilot policy: keep verified complete arms; restart all partial/missing/incompatible arms.")

The preflight checks GPU access, a tiny GPT-2 forward and the review pipeline.
Packages stay isolated from the notebook kernel while preserving Colab's CUDA
stack. Select a T4 runtime if CUDA is unavailable. The cached dataset and original
pretrained model can be reused from Drive. Failed stages are recorded in the ZIP.

In [ ]:
runtime_code = r"""
import importlib.metadata, json, platform, sys
from pathlib import Path
import torch
from packaging.version import Version
from dsa.models.backbone import BackboneAdapter
assert (3, 10) <= sys.version_info[:2] < (3, 14), "Use a Python 3.10–3.13 runtime."
assert Version(torch.__version__.split('+')[0]) >= Version('2.7'), "Use PyTorch >=2.7."
requested_device, output = sys.argv[1:]
probe = {'requested_device': requested_device, 'python': platform.python_version(),
         'torch': torch.__version__, 'cuda_build': torch.version.cuda,
         'cuda_available': torch.cuda.is_available(), 'cuda_device_count': torch.cuda.device_count()}
print(json.dumps({'runtime_probe': probe}, indent=2), flush=True)
if requested_device == 'cuda' and not probe['cuda_available']:
    raise RuntimeError('CUDA is unavailable in this runtime. In Colab choose Runtime > Change runtime type > T4 GPU, '
                       'save/connect, then rerun setup. If a GPU is already allocated, inspect the runtime probe '
                       'and nvidia-smi output before changing packages.')
torch.set_num_threads(1)
model = BackboneAdapter.tiny(259).model.to(requested_device).eval()
with torch.no_grad():
    result = model(input_ids=torch.tensor([[3, 4, 5]], device=requested_device), use_cache=False)
assert bool(torch.isfinite(result.logits).all()), 'Nonfinite GPT-2 forward'
report = {'python': platform.python_version(), 'device': requested_device,
          'cuda_available': torch.cuda.is_available(), 'cuda': torch.version.cuda,
          'gpu': torch.cuda.get_device_name() if torch.cuda.is_available() else None,
          'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
          'packages': {name: importlib.metadata.version(name) for name in
                       ('torch', 'transformers', 'tokenizers', 'huggingface_hub', 'numpy')},
          'tiny_gpt2_forward_passed': True}
Path(output).write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))
"""
GPU_READY = stage("runtime-preflight", "-c", runtime_code, DEVICE, REVIEW_ROOT / "runtime.json")
CHECKS_READY = stage("review-correctness", "-m", "unittest", "discover", "-s", "tests", "-p", "test_pilot_*.py", "-q")
print("Preflight complete; inspect recorded failures before interpreting results.")
MODEL_CHECKS_READY = stage("memory-injection-correctness", "-m", "unittest", "discover", "-s", "tests", "-p", "test_flt_memory.py", "-q")
GPU_READY = GPU_READY and CHECKS_READY and MODEL_CHECKS_READY

The original checksum-selected prepared data is reused when available. If it is
missing, the verified downloader/preparation workflow rebuilds the 32-row subset,
reusing cached source files. Source-quality exclusions remain off.

In [ ]:
DATA_READY = True
if RUN_MODE == "train_pilot" or not (PREPARED_DRIVE / "manifest.json").is_file():
    DATA_READY = stage("prepare-data", "scripts/prepare_dsa_pilot.py",
        "--storage-dir", PROJECT_DIR / "corpus", "--output-dir", PREPARED_DRIVE,
        "--train-limit", TRAIN_LIMIT, "--loops", LOOPS, "--seed", SEED,
        "--tokenizer-model", BACKBONE, "--revision", MODEL_REVISION)
if DATA_READY:
    try:
        shutil.copytree(PREPARED_DRIVE, PREPARED_LOCAL, dirs_exist_ok=True)
        prepared_manifest = json.loads((PREPARED_LOCAL / "manifest.json").read_text())
        record_stage("stage-cached-inputs", "passed", prepared_dir=str(PREPARED_DRIVE))
        print("Prepared records:", {name: entry["records"] for name, entry in prepared_manifest["splits"].items()})
    except Exception as error:
        DATA_READY = False
        record_stage("stage-cached-inputs", "failed", error=str(error))
        print("Input staging failed:", error)
# Selection uses the same pinned source, seed and 32-row limit. If the original
# prepared cache is available, independently check that K=8 selected the same IDs.
if DATA_READY:
    reference = PROJECT_DIR / "prepared" / "ba4e69315086-train32-k16-s7" / "train.tokenized.jsonl"
    if reference.is_file() and TRAIN_LIMIT == 32 and SEED == 7:
        old_ids = [json.loads(line)["sample_id"] for line in reference.read_text().splitlines() if line.strip()]
        current = PREPARED_LOCAL / prepared_manifest["splits"]["train"]["path"]
        new_ids = [json.loads(line)["sample_id"] for line in current.read_text().splitlines() if line.strip()]
        DATA_READY = old_ids == new_ids
        record_stage("same-original-training-selection", "passed" if DATA_READY else "failed")
        if not DATA_READY: print("Training selection differs from the original pilot; training is skipped.")

In [ ]:
SMOKE_READY = RUN_MODE == "review_existing"
if RUN_MODE != "review_existing" and GPU_READY and DATA_READY:
    SMOKE_READY = stage("original-checkpoint-smoke", "scripts/smoke_pretrained_dsa.py", "--backbone", BACKBONE,
          "--revision", MODEL_REVISION, "--device", DEVICE, "--output", SMOKE_REPORT)
else:
    print("Original smoke generation skipped.")

The screen uses the same initial model parameters in all conditions, records
answer gradients and residual norms before normalization, and performs no optimizer
updates. It reuses verified completed screen cases after reconnecting.

Each training condition then reaches **200 updates at K=8**, or is recorded as
failed. Initialization, selected rows, runtime and sample order are matched.
Loss weights, optimizer and clipping stay fixed so the recurrence comparison is
interpretable. Partial runs restart; finished compatible arms are reused.

In [ ]:
SCREEN_READY = RUN_MODE == "review_existing"
if RUN_MODE != "review_existing" and GPU_READY and DATA_READY and SMOKE_READY:
    SCREEN_READY = stage("initialization-gradient-screen", "scripts/screen_recurrence.py",
        "--prepared-dir", PREPARED_LOCAL, "--output-dir", REVIEW_ROOT / "initialization-screen",
        "--device", DEVICE, "--smoke-report", SMOKE_REPORT,
        "--conditions", json.dumps({arm: CONDITIONS[arm] for arm in ARMS}),
        "--limit", SCREEN_LIMIT, "--depths", *SCREEN_DEPTHS, "--seed", SEED, "--workspace", WORKSPACE)
    if SCREEN_READY:
        screen_report = json.loads((REVIEW_ROOT / "initialization-screen/report.json").read_text())
        SCREEN_READY = screen_report["finite"] and screen_report["model_state_unchanged"]
        print("Initialization gradient ratios (FLT / standard):", screen_report["flt_to_standard_ratios"])
        print("These probes describe conditioning; they do not establish an accuracy advantage.")


In [ ]:
training_reports, arm_checkpoints, READY_ARMS = {}, {}, []
initial_sha = globals().get("screen_report", {}).get("initial_state_sha256")
BACKUP_DOWNLOADS_STARTED = set()

def download_full_checkpoints():
    from google.colab import files
    paths = {arm: Path(result["local_full_backup"]) for arm, result in arm_checkpoints.items()
             if Path(result["local_full_backup"]).is_file()}
    for arm, path in paths.items():
        if arm in BACKUP_DOWNLOADS_STARTED: continue
        print(f"FULL BACKUP {arm}: {path.stat().st_size / 1e9:.2f} GB. Keep this ZIP and wait for the download to finish.")
        files.download(str(path))
        BACKUP_DOWNLOADS_STARTED.add(arm)
        record_stage(f"backup-download-{arm}", "passed", request_started_only=True)
    return bool(paths) and set(paths) <= BACKUP_DOWNLOADS_STARTED

for arm in ARMS:
    run_dir = RUN_ROOT / arm
    if RUN_MODE == "review_existing":
        READY_ARMS.append(arm)
    elif GPU_READY and DATA_READY and SMOKE_READY and SCREEN_READY:
        out = REVIEW_ROOT / arm / "checkpoint-status.json"
        args = ["scripts/ensure_pilot_arm.py", "--prepared-dir", PREPARED_LOCAL,
            "--working-dir", run_dir, "--legacy-dir", LEGACY_RUN_ROOT / arm,
            "--backup-dir", CHECKPOINT_BACKUP_DIR, "--report-path", out, "--arm", arm,
            "--steps", STEPS, "--device", DEVICE, "--smoke-report", SMOKE_REPORT,
            "--seed", SEED, "--workspace", WORKSPACE, "--learning-rate", LEARNING_RATE,
            "--gradient-accumulation", GRADIENT_ACCUMULATION,
            "--write-mode", CONDITIONS[arm]["write_mode"], "--recurrence-mode", CONDITIONS[arm]["recurrence_mode"]]
        if TEMPLATE_RUN_DIR: args += ["--template-dir", TEMPLATE_RUN_DIR]
        if initial_sha: args += ["--expected-initial-sha", initial_sha]
        if RUN_MODE == "restart_all": args += ["--force-restart"]
        if stage(f"ensure-{arm}", *args):
            result = json.loads(out.read_text())
            arm_checkpoints[arm] = result
            initial_sha = result["initial_state_sha256"]
            READY_ARMS.append(arm)
            if DOWNLOAD_FULL_CHECKPOINTS:
                try: download_full_checkpoints()
                except Exception as error:
                    record_stage(f"backup-download-{arm}", "failed", error=str(error))
                    print("Full backup remains on VM/Drive; final cell retries:", error)
            record_stage(f"drive-backup-{arm}", "failed" if result["backup_error"] else "passed",
                         error=result["backup_error"], mounted_readback_only=True)
    if arm in READY_ARMS:
        report_path = run_dir / "train_report.json"
        training_reports[arm] = json.loads(report_path.read_text()) if report_path.exists() else {
            "step": json.loads((run_dir / "latest.json").read_text())["step"]}
        print(f"{arm}: ready at step {training_reports[arm]['step']}")
# Bind review outputs to the actual checkpoint bytes. A restart gets a fresh review
# location whenever its state differs; stale predictions never masquerade as new.
if arm_checkpoints:
    checkpoint_identity = {arm: {name: entry["sha256"] for name, entry in result["backup_files"].items()
        if name.startswith("checkpoint-")} for arm, result in arm_checkpoints.items()}
    checkpoint_hash = hashlib.sha256(json.dumps(checkpoint_identity, sort_keys=True).encode()).hexdigest()[:12]
    REVIEW_NAME = BASE_REVIEW_NAME + "-" + checkpoint_hash
    selected_review = REPORT_ROOT / REVIEW_NAME
    selected_review.mkdir(parents=True, exist_ok=True)
    for path in REVIEW_ROOT.rglob("*"):
        relative = path.relative_to(REVIEW_ROOT)
        if selected_review != REVIEW_ROOT and path.is_file() and (relative.parts[0] == "logs" or path.name in {
                "checkpoint-status.json", "runtime.json", "session_status.json", "protocol.json"} or relative.parts[0] == "initialization-screen"):
            target = selected_review / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
    REVIEW_ROOT = selected_review
    status_path = REVIEW_ROOT / "session_status.json"
    config["REVIEW_NAME"] = REVIEW_NAME
    (REVIEW_ROOT / "protocol.json").write_text(json.dumps(config, indent=2) + "\n")
    print("Checkpoint-bound review:", REVIEW_ROOT)

FULL_BACKUP_DOWNLOAD_STARTED = bool(arm_checkpoints) and set(arm_checkpoints) <= BACKUP_DOWNLOADS_STARTED

Question-only validation uses the original scorer on the first 128 source-order
questions by default. Set `VALID_LIMIT=0` before this review for all 500. Saved
predictions resume within the requested evaluation. Earlier 32-question reports
are preserved and included in the final ZIP. New reports go under this review
name, so the original pilot summary remains intact. Test answers stay reserved.


In [ ]:
def checkpoint_step(arm):
    return json.loads((RUN_ROOT / arm / "latest.json").read_text())["step"]
def evaluation_dir(arm, split, budget, limit):
    return REVIEW_ROOT / arm / f"{split if split == 'test' else 'validation'}-step{checkpoint_step(arm)}-k{budget}-n{limit}-g{MAX_NEW_TOKENS}"
validation_reports = []
if GPU_READY and DATA_READY:
    for arm in READY_ARMS:
        for budget in EVAL_BUDGETS:
            out = evaluation_dir(arm, "valid", budget, VALID_LIMIT)
            if stage(f"validation-{arm}-k{budget}", "scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                     "--run-dir", RUN_ROOT / arm, "--output-dir", out, "--split", "valid",
                     "--loop-budget", budget, "--limit", VALID_LIMIT, "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE):
                validation_reports.append({"arm": arm, **json.loads((out / "report.json").read_text())})

## Training behavior, memory interventions and gradients

All training questions receive question-only generation. The first eight prepared
training rows are paired under normal inference, K=1/2/4/8/16, disabled/always-on
writes, and zeroed final anchor/workspace/both. Zeroing affects only the answer
readout; it preserves the computed latent trajectory and sequence positions.
These interventions can be out of distribution and do not prove semantic reasoning.

Every training example also gets a deterministic teacher-forced loss and latent
state profile, including input/memory scales, writes and teacher milestone
alignment. The first four training rows receive per-loop gradient probes. At the
original budget, weighted answer/CoT/gate/alignment contributions use the same
forward graph. Controller/projection gradient cosines check competing directions.
No clipping or optimizer updates occur; model-state checksums must remain equal.


In [ ]:
diagnostic_reports = {}
if GPU_READY and DATA_READY:
    for arm in READY_ARMS:
        out = REVIEW_ROOT / arm / "diagnostics"
        if stage(f"diagnostics-{arm}", "scripts/diagnose_dsa_pilot.py", "--prepared-dir", PREPARED_LOCAL,
                 "--run-dir", RUN_ROOT / arm, "--output-dir", out, "--device", DEVICE,
                 "--gradient-limit", GRADIENT_LIMIT, "--probe-limit", PROBE_LIMIT,
                 "--depth-grid", *DEPTH_GRID, "--max-new-tokens", MAX_NEW_TOKENS, "--comprehensive"):
            diagnostic_reports[arm] = json.loads((out / "report.json").read_text())

## Explicit reasoning controls

These controls generate from question text through the explicit language route,
without recurrent memory. Compare the original cached pretrained model with the
shared backbone after each arm's training, on the same first eight training and
first eight validation questions. They use a larger 192-token cap for CoT, so
this is a diagnostic comparison with different compute budgets. The original
model is loaded with `local_files_only=True`; a missing cache produces a recorded
failure rather than a new model download. Test questions are never read.


In [ ]:
control_reports = {}
if GPU_READY and DATA_READY:
    conditions = [("original", None)] + [(arm, RUN_ROOT / arm) for arm in READY_ARMS]
    for name, run_dir in conditions:
        out = REVIEW_ROOT / "original-explicit-control" if run_dir is None else REVIEW_ROOT / name / "explicit-control"
        args = ["scripts/pilot_explicit_controls.py", "--prepared-dir", PREPARED_LOCAL,
                "--output-dir", out, "--device", DEVICE, "--limit", CONTROL_LIMIT,
                "--max-new-tokens", COT_MAX_NEW_TOKENS]
        if run_dir is not None: args += ["--run-dir", run_dir]
        if stage(f"explicit-control-{name}", *args): control_reports[name] = json.loads((out / "report.json").read_text())

## Final held-out evaluation — leave disabled during development

Freeze the training setup, checkpoint choice, decoding settings and loop budgets
using validation before enabling this cell. It evaluates **every one of the 1,319
test questions** for every declared arm/budget with the pinned LOTUS scorer. Test
answers do not select checkpoints or tune any setting. A small-subset training
run still remains a pilot even if it evaluates the full test split.


In [ ]:
RUN_TEST = False  # Enable only after freezing the complete evaluation protocol.
final_reports = []
if RUN_TEST:
    for arm in ARMS:
        for budget in EVAL_BUDGETS:
            out = evaluation_dir(arm, "test", budget, 0)
            run_project("scripts/evaluate_dsa.py", "--prepared-dir", PREPARED_LOCAL,
                        "--run-dir", RUN_ROOT / arm, "--output-dir", out,
                        "--split", "test", "--loop-budget", budget, "--limit", 0,
                        "--max-new-tokens", MAX_NEW_TOKENS, "--device", DEVICE, "--final-test")
            result = json.loads((out / "report.json").read_text())
            assert result["total"] == 1319 and result["all_source_examples_retained"]
            final_reports.append({"arm": arm, **result})
else:
    print("Held-out test evaluation is disabled. Use validation for pilot decisions.")


In [ ]:
print(f"{'arm':12} {'split':6} {'K':>3} {'n':>5} {'accuracy':>9} {'invalid':>9} {'sec/q':>9}")
for report in globals().get("validation_reports", []) + globals().get("final_reports", []):
    print(f"{report['arm']:12} {report['split']:6} {report['loop_budget']:3d} {report['total']:5d} "
          f"{report['accuracy']:9.3%} {report['invalid_answer_rate']:9.3%} {report['average_latency_seconds']:9.3f}")
training_profiles = {}
for arm, report in training_reports.items():
    journal = RUN_ROOT / arm / "metrics.jsonl"
    metrics = []
    if journal.exists():
        with journal.open() as handle:
            for line in handle:
                if not line.strip(): continue
                try: metrics.append(json.loads(line))
                except json.JSONDecodeError:
                    print("Incomplete/corrupt metric line:", journal)
    measured_seconds = sum(row["seconds"] for row in metrics)
    profile = {"completed_steps": report["step"], "measured_update_seconds": measured_seconds,
               "updates_per_second": len(metrics) / measured_seconds if measured_seconds else None,
               "peak_process_rss_bytes": max((row["peak_process_rss_bytes"] for row in metrics), default=0),
               "peak_cuda_allocated_bytes": max((row["peak_cuda_allocated_bytes"] or 0 for row in metrics), default=0),
               "first_total_loss": metrics[0]["total"] if metrics else None,
               "last_total_loss": metrics[-1]["total"] if metrics else None,
               "mean_write_rate": sum(row["write_rate"] for row in metrics) / len(metrics) if metrics else None}
    training_profiles[arm] = profile
    print(arm, "training profile:", json.dumps(profile, indent=2))
for arm, result in globals().get("diagnostic_reports", {}).items():
    print(arm, "training generation:", result["prediction_reports"]["normal_training_set"])
for name, result in globals().get("control_reports", {}).items():
    print(name, "explicit control:", result["splits"])
prepared_path = PREPARED_LOCAL / "manifest.json"
summary = {"config": config, "training_profiles": training_profiles,
           "prepared_manifest_sha256": hashlib.sha256(prepared_path.read_bytes()).hexdigest() if prepared_path.exists() else None,
           "training": training_reports, "validation": globals().get("validation_reports", []),
           "final_test": globals().get("final_reports", []), "stage_status": globals().get("STAGE_STATUS", {}),
           "purpose": "pilot diagnosis before a larger protocol; no comparative research claim"}
summary_path = globals().get("REVIEW_ROOT", REPORT_ROOT) / "notebook-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n")
print("Persistent summary:", summary_path)
print("Continue to the ZIP export cell, including after recorded GPU-stage failures.")


## One review ZIP and four independent checkpoint backups

Hand back the **small review ZIP**. It includes all four conditions, the
initialization gradient screen, raw residual norms before normalization, losses,
latent/gate/teacher profiles, per-loss/per-loop gradients, paired interventions,
explicit controls, provenance, logs, file hashes and completeness flags.
`scale_readiness.json` compares recurrence architectures at fixed write policy
and write policies within each architecture. Its memorization threshold is a
heuristic; it does not automatically approve scaling.

Keep the **four full checkpoint ZIPs** downloaded immediately after each arm.
They contain weights, tokenizer, optimizer/scheduler/RNG and journals. Each is
also copied as a checksummed file to Drive. They are intentionally separate to
avoid a single multi-GB browser bundle of all four models.

In [ ]:
from google.colab import files
BUNDLE_PATH = PROJECT_DIR / "bundles" / f"{RUN_NAME}-{REVIEW_NAME}.zip"
if stage("review-export", "scripts/package_pilot_review.py", "--project-dir", PROJECT_DIR,
            "--run-name", RUN_NAME, "--review-name", REVIEW_NAME, "--run-root", RUN_ROOT,
            "--prepared-dir", PREPARED_DRIVE, "--output", BUNDLE_PATH, "--arms", *ARMS):
    print("SMALL REVIEW ZIP:", BUNDLE_PATH)
    print("Hand this ZIP back for analysis. Keep the four separate full checkpoint ZIPs as your backups.")
    files.download(str(BUNDLE_PATH))
if DOWNLOAD_FULL_CHECKPOINTS and not FULL_BACKUP_DOWNLOAD_STARTED:
    try: FULL_BACKUP_DOWNLOAD_STARTED = download_full_checkpoints()
    except Exception as error: print("Full archives remain on the VM and Drive; download error:", error)
print("Recorded failed stages:", {name: entry for name, entry in STAGE_STATUS.items() if entry['status'] != 'passed'})